<a href="https://colab.research.google.com/github/lin031029tom-bit/Ai-with-government/blob/main/road_safety_dissertation_coding_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Road Safety Dissertation Coding

This clean notebook checks out the validated data-publication snapshot and runs the repository's one-command dissertation reproduction. The command automatically extracts and validates the published 503,475-row dataset, executes full-data modelling, uncertainty, temporal validation and robustness checks, and compares the generated results with the verified reference tables. The task is retrospective severity classification conditional on a collision having occurred and been reported; it does not forecast future collision occurrence, time or location.

## 1. Environment setup

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path("/content/Ai-with-government")
repo_url = "https://github.com/lin031029tom-bit/Ai-with-government.git"
repo_ref = "2148d0950fe026f66dd5ea1810807af32ce91af1"

if repo.exists():
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", "--tags"], check=True)
else:
    subprocess.run(["git", "clone", repo_url, str(repo)], check=True)

subprocess.run(["git", "-C", str(repo), "fetch", "origin", repo_ref], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
resolved_ref = subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
assert resolved_ref == repo_ref, f"Expected {repo_ref}, checked out {resolved_ref}"
print("Working directory:", Path.cwd())
print("Pinned code commit:", resolved_ref)

## 2. Validate required files

In [ ]:
from pathlib import Path
import hashlib

archive_path = Path("published_data/analysis_ready_road_safety.csv.gz")
expected_archive_sha256 = "8efbdd94ad028113facbec818dc3e95c9a98621e6739450a56a21d7b995b00c7"
required_files = [
    Path("analysis_schema.py"),
    Path("reproduce_dissertation.py"),
    Path("validate_analysis_ready_data.py"),
    Path("verify_dissertation_results.py"),
    Path("road_safety_dissertation_coding.py"),
    archive_path,
    Path("example_results/tables/table_4_3_model_performance_2024_test.csv"),
]

file_status = {str(path): path.exists() for path in required_files}
print("Required files:", file_status)

missing_files = [path for path in required_files if not path.exists()]
assert not missing_files, f"Required files not found: {missing_files}"
archive_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
assert archive_sha256 == expected_archive_sha256, (archive_sha256, expected_archive_sha256)
print("Published archive SHA-256:", archive_sha256)

## 3. Run full analysis, uncertainty, temporal validation and robustness checks

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "reproduce_dissertation.py",
    ],
    check=True,
)
print("Complete dissertation reproduction passed.")

## 4. Review generated tables

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

tables_dir = Path("road_safety_coding_outputs/tables")
for table_path in sorted(tables_dir.glob("*.csv")):
    print(f"\n{table_path.name}")
    display(pd.read_csv(table_path))

## 5. Review generated figures

In [ ]:
from pathlib import Path
from IPython.display import Image, display

figures_dir = Path("road_safety_coding_outputs/figures")
for figure_path in sorted(figures_dir.glob("*.png")):
    print(figure_path.name)
    display(Image(filename=str(figure_path)))